# 00a - CIFAR-100 self-train + one-pass extract (pipeline DEBUG backbone)

No released CIFAR-100 checkpoint exists, so we own the whole stack: train a ResNet-50,
then extract scores + penultimate embeddings from that same model. The checkpoint->score
**reproduction gate is N/A** here; a **self-consistency check** replaces it.

**THIN-REGIME CAVEAT (binding):** 100 classes -> 50/50 class split = only 50 fit / 50 held-out
for g_theta. CIFAR-100 is **pipeline debug ONLY, never the gate verdict** (that is Pl@ntNet).

**Colab survival:** the checkpoint is written to **Google Drive every epoch** and training
**resumes** from it - a killed session loses nothing. Recipe follows CCC cifar_utils
(resnet50 IMAGENET1K_V2 -> fc100, Adam lr 1e-4). Numbers are NOT compared to CCC's.


## 1. GPU check


In [ ]:
import subprocess
o = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(o.stdout if o.returncode==0 else 'WARNING: no GPU - training will be very slow on CPU.')


## 2. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

NUM_EPOCHS = 15                 # CCC uses 30; 15 is enough to debug the pipeline
LR         = 1e-4
BATCH      = 64
NUM_WORKERS= 2
VAL_FRAC   = 0.1                # slice of train used for best-epoch selection
PER_CLASS_TRAIN_QUOTA = 50      # train images/class for DESCRIPTORS (from train only)
SEED = 42

DATA_ROOT  = f'{DRIVE_ROOT}/data/cifar100'            # torchvision downloads here
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints/cifar100'      # <-- checkpoints saved HERE (Drive)
EMB_DIR    = f'{DRIVE_ROOT}/embeddings/cifar100/resnet50_self'
# =======================================================================
print('CKPT_DIR (Drive) =', CKPT_DIR)


## 3. Mount Drive + repo + pinned env + seed + versions


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
REPO_ROOT = os.getcwd()
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
os.environ['PYTHONPATH'] = REPO_ROOT + os.pathsep + os.environ.get('PYTHONPATH','')
os.environ['PYTHONUTF8'] = '1'
import os as _os; [_os.makedirs(d, exist_ok=True) for d in (DATA_ROOT, CKPT_DIR, EMB_DIR)]
from pcc.utils.seed import set_seed
from pcc.utils.device import get_device, gpu_name
from pcc.utils.io import environment_stamp
set_seed(SEED); DEVICE = get_device()
print('GPU:', gpu_name()); print('env:', environment_stamp()['packages'])


## 4. Data - CIFAR-100 (torchvision, small). Train / val-slice / test.


In [ ]:
import numpy as np, torch
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR100
from pcc.extract.train_cifar import cifar_transforms
train_tfm, test_tfm = cifar_transforms()

train_full = CIFAR100(DATA_ROOT, train=True,  download=True, transform=train_tfm)
test_set   = CIFAR100(DATA_ROOT, train=False, download=True, transform=test_tfm)

# deterministic val slice of TRAIN for best-epoch selection
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(train_full))
n_val = int(VAL_FRAC * len(train_full))
val_idx, fit_idx = perm[:n_val], perm[n_val:]
train_loader = DataLoader(Subset(train_full, fit_idx.tolist()), batch_size=BATCH, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(Subset(train_full, val_idx.tolist()), batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)
print(f'train fit={len(fit_idx)} val={len(val_idx)} test={len(test_set)}')


## 5. Train (resumable; checkpoint -> Drive every epoch)


In [ ]:
from pcc.extract.train_cifar import build_cifar_resnet50, train_cifar100
model = build_cifar_resnet50(num_classes=100, device=DEVICE)
best_path = train_cifar100(model, train_loader, val_loader, ckpt_dir=CKPT_DIR,
                           device=DEVICE, num_epochs=NUM_EPOCHS, lr=LR, resume=True, seed=SEED)
print('best checkpoint on Drive:', best_path)
# load best weights for extraction
import torch
model.load_state_dict(torch.load(best_path, map_location=DEVICE)['model']); model.eval()


## 6. Extract logits + penultimate embeddings (deterministic test transform)
Test set -> scores for cal/eval. Per-class TRAIN subset (test transform, no aug) ->
embeddings for descriptors (descriptors come from TRAINING data only, Sec 6.3).


In [ ]:
from pcc.extract.backbones import forward_logits_and_embeddings
import numpy as np, os
from torchvision.datasets import CIFAR100

# (a) TEST set: scores + embeddings (shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)
te_softmax, te_labels, te_emb, te_logits = forward_logits_and_embeddings(
    model, test_loader, DEVICE, capture_embeddings=True, return_logits=True)
np.savez(os.path.join(EMB_DIR,'test.npz'), softmax=te_softmax, labels=te_labels,
         embeddings=te_emb, logits=te_logits)

# (b) per-class TRAIN subset (deterministic test transform) -> descriptor embeddings
train_det = CIFAR100(DATA_ROOT, train=True, download=False, transform=test_tfm)
tl = np.array(train_det.targets)
sel = []
for c in range(100):
    idx_c = np.where(tl==c)[0]
    sel.extend(rng.choice(idx_c, min(PER_CLASS_TRAIN_QUOTA, len(idx_c)), replace=False).tolist())
sub_loader = DataLoader(Subset(train_det, sel), batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)
tr_softmax, tr_labels, tr_emb, tr_logits = forward_logits_and_embeddings(
    model, sub_loader, DEVICE, capture_embeddings=True, return_logits=True)
np.savez(os.path.join(EMB_DIR,'train_subset.npz'), softmax=tr_softmax, labels=tr_labels,
         embeddings=tr_emb, logits=tr_logits, quota=PER_CLASS_TRAIN_QUOTA)
print('saved test + train_subset embeddings to', EMB_DIR)


## 7. Self-consistency check (replaces the reproduction gate) + report/marker


In [ ]:
import time, json, os
from pcc.extract.train_cifar import self_consistency_check
from pcc.eval.score_repro import sha256_file
from pcc.utils.io import write_report

res = self_consistency_check(model, te_emb, te_logits, te_softmax, te_labels)
print(json.dumps(res, indent=2))

report = write_report('pcc/reports','00a_cifar100_selfconsistency',
    hypothesis='self-trained CIFAR-100 resnet50: extraction is internally consistent',
    pass_criteria='softmax recompute maxdiff<=1e-6; penultimate@fc->logits maxdiff<=1e-3; sane val acc',
    config=dict(num_epochs=NUM_EPOCHS, lr=LR, batch=BATCH, val_frac=VAL_FRAC,
                per_class_train_quota=PER_CLASS_TRAIN_QUOTA, recipe='CCC cifar_utils (no normalize)'),
    seed=SEED, results={**res, 'checkpoint_sha256': sha256_file(best_path)},
    conclusion=res['verdict'], started_at=time.time())
print('report:', report)

MARK = f'{DRIVE_ROOT}/gates/SELFCONSISTENT_cifar100.json'
os.makedirs(os.path.dirname(MARK), exist_ok=True)
if res['verdict']=='PASS':
    json.dump({'dataset':'cifar100','results':res,'best_ckpt':best_path,'emb_dir':EMB_DIR},
              open(MARK,'w'), indent=2)
    print('SELF-CONSISTENCY PASS - pipeline may use these CIFAR-100 embeddings for Phase 0/1 debug.')
else:
    if os.path.exists(MARK): os.remove(MARK)
    raise SystemExit('SELF-CONSISTENCY FAILED - hook/softmax/logit path bug. Report results above.')
